# Recovering a Simulated Galaxy

This notebook simulates a galaxy with known parameters, fits it with the
public BayMorph workflow, and checks that the posterior recovers the truth.

The run is deliberately small so it finishes quickly. Matching the truth once
is a smoke test, not a convergence check.

In [ ]:
from astropy.table import Table

from baymorph import BayMorphRunner
from baymorph.plotting import corner_plot, plot_log_image

## Configuration

One YAML file defines the whole analysis: the truth, the PSF, the priors, the
likelihood and the sampler.

In [ ]:
runner = BayMorphRunner("configs/simulated_galaxy.yaml")

print(f"Resolved configuration: {runner.config_path}")

## Simulate

`simulate()` creates the synthetic observation and writes it as FITS.

In [ ]:
runner.simulate()

In [ ]:
plot_log_image(
    runner.simulation.image,
    title="Simulated galaxy",
    colorbar_label="Counts",
    show=True,
)

## Prepare and fit

`prepare_image()` builds the cutout, background, noise and mask. `fit()` then
runs the sampler. The configuration declares an analytic Gaussian PSF, so
there is no PSF to build.

The fit is the slow cell.

In [ ]:
runner.prepare_image()

fit = runner.fit()

## Parameter recovery

Compare each truth with the `q16`-`q84` credible interval. This run is
deliberately small, so offsets of one to two sigma are normal and are not by
themselves a failure.

In [ ]:
truth = runner.simulation.truth

recovery_table = Table(
    rows=[
        {
            "parameter": name,
            "truth": truth[name],
            **fit.posterior.summary(name),
        }
        for name in fit.posterior.parameter_names
        if name in truth
    ]
)

recovery_table

## Diagnostics

`diagnostics()` writes the corner plot, residual maps and radial profile as
PDF files in the run directory.

In [ ]:
runner.diagnostics()

print(f"Diagnostics written to: {runner.run_dir}")

The saved corner plot already marks the truth in orange. The cell below
draws the same plot inline.

In [ ]:
# This configuration's cutout spans the whole simulated image, so the truth
# positions need no cutout offset here.
corner_figure, _, _ = corner_plot(
    fit.posterior.equal_weight_samples.T,
    parameter_labels=fit.posterior.parameter_names,
    truths=[
        truth.get(name)
        for name in fit.posterior.parameter_names
    ],
    title="Simulated galaxy",
)

## Summary

In [ ]:
inference = fit.posterior

print(f"Backend: {inference.provenance.get('backend', 'unknown')}")
print(f"Reduced chi2: {fit.reduced_chi2:.4f} (ndof = {fit.ndof})")

if inference.logz is None:
    print("Evidence: unavailable (posterior-only backend)")
else:
    print(f"logZ: {inference.logz:.4f}")

fit.summary()